# Tools — 工具定义与调用

对标文档：LangChain Core Components > Tools

In [6]:
from langchain_learning.llm import get_llm
from langchain_core.tools import tool

llm = get_llm()

**术语：**
- **Tool（工具）** = 给 LLM 调用的函数，LLM 可以决定何时调用、传什么参数
- **@tool** = 装饰器，把普通 Python 函数变成 LLM 可调用的工具
- **bind_tools** = 把工具注册到 LLM 上，告诉 LLM 有哪些工具可用

### 1. @tool 装饰器——定义第一个工具

In [7]:
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气情况"""
    return f"{city} 今天晴天，气温 22-28°C"

print(get_weather.name)
print(get_weather.invoke({"city": "北京"}))

get_weather
北京 今天晴天，气温 22-28°C


**术语：**
- **@tool** = 装饰器，自动根据函数签名（函数名、参数、类型注解、docstring）生成工具的 schema
- **name** = 工具名，默认就是函数名
- **invoke** = 调用工具，参数以字典形式传入

### 2. 带类型注解的工具参数

In [8]:
@tool
def calculate(expr: str) -> float:
    """计算数学表达式并返回结果"""
    try:
        return eval(expr)
    except:
        return 0.0

@tool
def search_news(keyword: str, limit: int = 5) -> list:
    """搜索新闻，返回指定数量的结果"""
    results = [f"{keyword} 新闻{i}" for i in range(limit)]
    return results

print(calculate.invoke({"expr": "2 + 3 * 4"}))
print(search_news.invoke({"keyword": "AI", "limit": 3}))

14
['AI 新闻0', 'AI 新闻1', 'AI 新闻2']


**术语：**
- **类型注解** = `city: str`、`limit: int = 5`，告诉 LLM 参数的类型和默认值
- **docstring** = 函数开头的文字说明，LLM 通过它理解工具的作用

### 3. 把工具绑定到 LLM

In [9]:
tools = [get_weather, calculate, search_news]
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("北京天气怎么样？")
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_c4afufu8ikAv8zTY0Ha01832', 'type': 'tool_call'}]


**术语：**
- **bind_tools** = 把工具列表注册给 LLM，LLM 在回答时会判断是否需要调用
- **tool_calls** = LLM 返回的调用请求，包含工具名和参数，需要你自己执行
- **注意**：LLM 只是说应该调哪个工具，**不会自动执行**——需要你手动执行

### 4. 自动执行工具调用

In [10]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage("计算 (85 + 17) * 2 等于多少？")]

# 第一步：LLM 决定调用工具
response = llm_with_tools.invoke(messages)
messages.append(response)

print("LLM 决定调用:", response.tool_calls)

# 第二步：手动执行工具
for tool_call in response.tool_calls:
    selected_tool = {"calculate": calculate, "get_weather": get_weather, "search_news": search_news}[tool_call["name"]]
    tool_result = selected_tool.invoke(tool_call["args"])
    messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]))

# 第三步：把工具结果送回 LLM 生成最终回答
final = llm_with_tools.invoke(messages)
print("最终回答:", final.content)

LLM 决定调用: [{'name': 'calculate', 'args': {'expr': '(85 + 17) * 2'}, 'id': 'call_00_kf47nRTTMpt2lDyW1oaF6062', 'type': 'tool_call'}]
最终回答: 计算结果是：**(85 + 17) × 2 = 204**

计算步骤：
1. 先计算括号内的加法：85 + 17 = 102
2. 再乘以 2：102 × 2 = 204


**术语：**
- **ToolMessage** = 工具执行结果消息，需要带有 tool_call_id 关联到对应的调用请求
- **调用流程**：LLM 决定 → 手动执行 → 结果送回 LLM → 生成最终回答
- 这个模式在实际中通常用 AgentExecutor 自动处理，后面 Agent 章节会讲

完整流程：

完整流程：

```mermaid
graph LR
    A[用户问题] --> B[LLM + Tools]
    B --> C{需要调工具?}
    C -->|是| D[执行工具]
    D --> E[ToolMessage]
    E --> B
    C -->|否| F[最终回答]
    style B fill:#FFF3E0,stroke:#E65100
    style D fill:#E8F5E9,stroke:#2E7D32
```

### 5. 查看工具 schema

In [11]:
# 查看工具的参数定义（JSON Schema）
print("工具名:", calculate.name)
print("描述:", calculate.description)
print("参数 schema:", calculate.args)

工具名: calculate
描述: 计算数学表达式并返回结果
参数 schema: {'expr': {'title': 'Expr', 'type': 'string'}}


**术语：**
- **args** = 工具的参数定义，LLM 通过它知道要传什么参数
- **args_schema** = 也可以用 Pydantic 模型自定义参数结构（后面会讲）